# # 08 · Which genes explain the lipid changes?

Companion to the tutorial notebook [`08_multimodal_xgb_shap_go`](https://github.com/lamanno-epfl/lipidomics_tutorial_cajalcourse/tree/main/notebooks/level3).

**Kernel:** `cajal-lipidomics` · **Reads:** `06_clustered.h5ad` + Allen MERFISH + Gene Ontology files · **Writes:** `results/gene_*.csv`, `figures/fig_genes_explain_lipid_changes.(pdf|png)`

So far we've *described* what pregnancy does to the lipidome. Now we reach for a *mechanism*. Every lipid in a membrane was made, remodeled or broken down by an enzyme, and every enzyme is read off a gene. So if pregnancy reshapes a region's lipids, the genes that handle lipids there are probably doing something different. **Which genes track the lipid change closely enough to call them predictors?**

The second modality is the **Allen MERFISH** spatial transcriptome: different mice, different lab, different instrument. The only thing it shares with our MALDI data is **a map**: both were registered into the same Allen CCF. That shared coordinate system is the entire bridge, and we use it at two zoom levels:
- **Per pixel (fine):** each MALDI pixel borrows the genes and cell type of the MERFISH cells sitting within a small 3D ball around it.
- **Per region (coarse):** one lipid change and one gene profile per Allen region, joined on the region name, then a model ranks gene programs.

> Network note: the gene-ontology step converts gene symbols to Entrez IDs with one call to mygene.info. If you're offline, that step is skipped cleanly and everything else still runs.

In [ ]:
# ---- setup: the same first cell in every notebook ---------------------------------------
from lipid_project import paths
paths.ensure_dirs()
paths.summary()

import io, contextlib, os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad
import xgboost as xgb
from scipy.stats import pearsonr
from scipy.spatial import cKDTree
from scipy import sparse
from sklearn.decomposition import NMF
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

import cajal_lipidomics as cl
from cajal_lipidomics import plotting
from cajal_lipidomics.style import FS
cl.style.set_style()

SEED = 42
rng = np.random.default_rng(SEED)
print("ready. xgboost", xgb.__version__)

In [ ]:
# our notebook-06 output: pixels with uMAIA lipids, Allen region and lipizone
adata = ad.read_h5ad(paths.stage("clustered"))
U = np.asarray(adata.layers["umaia"], dtype=np.float64)              # uMAIA lipids: the measured values
# readable column labels; repeated names (one lipid, two adducts) get -1, -2 suffixes
lipid_names = ad.utils.make_index_unique(pd.Index(adata.var["lipid"].astype(str)))
obs = adata.obs    # short alias for the per-pixel table
is_ctrl = (obs["Condition"] == "naive").to_numpy()    # True for control pixels, False for pregnant
print("pixels x ions:", adata.shape, "| regions:", obs["acronym"].nunique(), "| lipizones:", obs["lipizone"].nunique())

# # Part A · Per pixel: borrow genes from the nearest MERFISH cells

We have two point clouds in the **same 3D CCF space**: our pixels at `(xccf, yccf, zccf)` and the MERFISH cells at `(x_ccf, y_ccf, z_ccf)`. For each pixel: which cells sit within a small ball around me? Average their genes and take a majority vote of their cell type.

We read only the cells near our plane: the parquet file is filtered while reading, on the anterior-posterior coordinate, with a small buffer. We drop vascular and immune cells (we never want to match tissue pixels to blood vessels).

In [ ]:
import pyarrow.parquet as pq
import pyarrow.compute as pc

MERFISH = paths.tutorial_data("merfish_plane.parquet")    # per-cell MERFISH near our plane (from the data bundle)
cols = pq.read_schema(MERFISH).names    # read just the column names, not the data, to decide what to load
gene_cols = [c for c in cols if c.startswith("ENSMUS")]                 # measured genes (Ensembl transcript ids)
# coordinates + cell-type labels (whichever exist in the file)
meta_cols = [c for c in ["x_ccf", "y_ccf", "z_ccf", "division", "class", "subclass", "supertype"] if c in cols]

ap = (obs["xccf"].min() - 0.1, obs["xccf"].max() + 0.1)               # our AP span, plus a 0.1 mm buffer
flt = (pc.field("x_ccf") >= ap[0]) & (pc.field("x_ccf") <= ap[1])       # pushed down into the file read
# load only the columns we need, only for cells inside the AP window
cells = pq.read_table(MERFISH, columns=meta_cols + gene_cols, filters=flt).to_pandas()
# drop blood-vessel and immune cells; reset_index renumbers rows 0..n-1
cells = cells[~cells["division"].astype(str).isin(["6 Vascular", "7 Immune"])].reset_index(drop=True)

print(f"AP window {ap[0]:.2f} to {ap[1]:.2f} | {len(cells):,} cells x {len(gene_cols)} genes | "
      f"{cells['subclass'].nunique()} cell-type subclasses")
print(f"cells z_ccf [{cells.z_ccf.min():.2f}, {cells.z_ccf.max():.2f}] vs pixels zccf "
      f"[{obs.zccf.min():.2f}, {obs.zccf.max():.2f}]  (they must overlap for any match to happen)")

# The only machinery is a **KD-tree** (`cKDTree`): it organises the cell coordinates once so that "all points within radius r of here" costs about log(N) instead of scanning every cell. One pixel by hand, at the paper's radius of 0.05 CCF units (≈ 50 µm, since 1 unit ≈ 1 mm):

In [ ]:
cell_xyz = cells[["x_ccf", "y_ccf", "z_ccf"]].to_numpy()    # the MERFISH point cloud, (cells, 3)
tree = cKDTree(cell_xyz)    # build the search tree once; every ball query reuses it
pix_xyz = obs[["xccf", "yccf", "zccf"]].to_numpy()    # our pixels in the same 3D space, (pixels, 3)

RADIUS_PAPER = 0.05    # CCF units are mm, so 0.05 = 50 µm
# Walk through pixels until one actually has cells inside its ball, so the demo isn't empty
pix = next((i for i in range(0, len(pix_xyz), 37) if tree.query_ball_point(pix_xyz[i], r=RADIUS_PAPER)), 0)
nbrs = tree.query_ball_point(pix_xyz[pix], r=RADIUS_PAPER)    # indices of every cell within the ball
G_cells = cells[gene_cols].to_numpy(np.float32)    # cells x genes expression matrix
print(f"pixel {pix} at CCF {pix_xyz[pix].round(2)} has {len(nbrs)} MERFISH cells within 50 µm")
if nbrs:
    # count each cell type inside the ball; the most frequent wins
    vals, counts = np.unique(cells["subclass"].astype(str).to_numpy()[nbrs], return_counts=True)
    print("borrowed gene vector length:", G_cells[nbrs].mean(0).shape[0], "| majority cell type:", vals[counts.argmax()])

# Now every pixel at once. A neat trick makes it fast: put the ball memberships into a sparse **pixel x cell** matrix `A` where each row holds `1/(number of cells in that ball)`. Then
- borrowed genes = `A @ genes` (each row is the average of its ball's cells)
- cell-type votes = `(A > 0) @ one_hot(cell type)` (each row counts its ball's cells per type)

Pixels with an empty ball stay unmatched (NaN).

In [ ]:
def match_pixels_to_cells(pix_xyz, radius, coverage_only=False):
    """Return (pixel x gene averages with NaN for unmatched, majority cell type per pixel or None).
    With coverage_only=True, just return the fraction of pixels that have at least one cell in their ball."""
    balls = tree.query_ball_point(pix_xyz, r=radius)                     # list of cell-index lists, one per pixel
    sizes = np.array([len(b) for b in balls])    # how many cells fell in each pixel's ball
    if coverage_only:
        return float((sizes > 0).mean())
    rows = np.repeat(np.arange(len(balls)), sizes)                       # pixel index for every (pixel, cell) pair
    # the matching cell index for every (pixel, cell) pair
    cols_ = np.concatenate([np.asarray(b, dtype=np.int64) for b in balls]) if sizes.sum() else np.array([], int)
    # A[p, c] = 1 if cell c is in pixel p's ball (sparse: almost all zeros)
    A = sparse.csr_matrix((np.ones(len(rows)), (rows, cols_)), shape=(len(balls), len(cell_xyz)))
    inv = sparse.diags(1.0 / np.maximum(sizes, 1))                       # 1/count per row (avoid divide-by-zero)
    # row-normalised A times genes = the average expression of each ball's cells
    genes = np.asarray((inv @ A) @ G_cells, dtype=np.float32)
    genes[sizes == 0] = np.nan    # pixels with an empty ball have no borrowed genes
    # list of cell types, and each cell's type as an integer code
    types, codes = np.unique(cells["subclass"].astype(str).to_numpy(), return_inverse=True)
    # one_hot[c, t] = 1 if cell c is of type t
    onehot = sparse.csr_matrix((np.ones(len(codes)), (np.arange(len(codes)), codes)), shape=(len(codes), len(types)))
    votes = (A @ onehot).tocsr()                                         # pixel x cell-type counts, kept sparse
    winner = np.asarray(votes.argmax(axis=1)).ravel()                    # most frequent cell type per pixel
    ctype = np.where(sizes > 0, types[winner], None)
    # pixel x gene table + one cell type per pixel
    return pd.DataFrame(genes, index=obs.index, columns=gene_cols), pd.Series(ctype, index=obs.index, name="celltype")


cov50 = match_pixels_to_cells(pix_xyz, 0.05, coverage_only=True)      # just count, to save memory
gdf, celltype = match_pixels_to_cells(pix_xyz, 0.10)    # the full match at 100 µm, used for everything below
print(f"pixels matched at  50 µm: {cov50:6.1%}")
print(f"pixels matched at 100 µm: {gdf.notna().any(axis=1).mean():6.1%}")

# Coverage is honest geometry, not a bug: our MALDI plane is one continuous sheet, while the MERFISH brain is a stack of **discrete** sections with gaps. Where our plane falls between two of them, no cell is close enough. Widening the ball to 100 µm reaches the neighbouring sections and lifts coverage, at the mild cost of averaging cells a bit farther away (which can blur sharp borders). We use 100 µm for the maps.

### A borrowed myelin gene next to a measured myelin lipid

The cleanest sanity check: a **myelin gene** should light up the same white matter as a **myelin lipid**. Gene: `ENSMUST00000102665`, the transcript of **Mog** (myelin oligodendrocyte glycoprotein), a textbook oligodendrocyte marker. Lipid: `HexCer 42:2`. Different mice, different instruments; if they agree, the shared-coordinate match worked.

In [ ]:
MYELIN_GENE = "ENSMUST00000102665"                    # Mog
# column of HexCer 42:2 if we annotated it, otherwise the first sphingolipid, otherwise column 0
j_lip = next((i for i, n in enumerate(lipid_names) if n.startswith("HexCer 42:2")),
             next((i for i, n in enumerate(lipid_names) if n.startswith(("HexCer", "SM", "Cer"))), 0))
MYELIN_LIPID = lipid_names[j_lip]

# the borrowed Mog value per pixel (NaN if unmatched)
gene_vec = gdf[MYELIN_GENE].to_numpy() if MYELIN_GENE in gdf else np.full(len(obs), np.nan)
lipid_vec = U[:, j_lip]    # the measured myelin lipid per pixel
matched = ~np.isnan(gene_vec)    # only pixels that borrowed genes can be compared
z, yv = obs["zccf"].to_numpy(), -obs["yccf"].to_numpy()    # plotting coordinates, dorsal up
secs = sorted(obs["SectionID"].unique())

fig, axes = plt.subplots(2, len(secs), figsize=(4.4 * len(secs), 7.2), constrained_layout=True)
for col, s in enumerate(secs):
    mm = (obs["SectionID"] == s).to_numpy() & matched    # this section's matched pixels
    lab = obs.loc[obs["SectionID"] == s, "Condition"].iloc[0]
    sc0 = axes[0, col].scatter(z[mm], yv[mm], c=gene_vec[mm], cmap="Reds", s=2.5, rasterized=True,
                               vmin=np.nanpercentile(gene_vec, 2), vmax=np.nanpercentile(gene_vec, 98))
    sc1 = axes[1, col].scatter(z[mm], yv[mm], c=lipid_vec[mm], cmap="plasma", s=2.5, rasterized=True,
                               vmin=np.percentile(lipid_vec[matched], 2), vmax=np.percentile(lipid_vec[matched], 98))
    axes[0, col].set_title(f"{lab}: borrowed Mog (MERFISH)", fontsize=FS["s"])
    axes[1, col].set_title(f"{lab}: {MYELIN_LIPID} (MALDI)", fontsize=FS["s"])
    for ax in axes[:, col]:
        cl.style.spatial_axes(ax); ax.set_xlim(z.min(), z.max()); ax.set_ylim(yv.min(), yv.max())
cl.style.lightweight_colorbar(sc0, list(axes[0]), label="Mog (borrowed)")
cl.style.lightweight_colorbar(sc1, list(axes[1]), label=MYELIN_LIPID)
plt.show()
# one number for how well the borrowed gene tracks the lipid
r_gl = pearsonr(gene_vec[matched], lipid_vec[matched])[0]
print(f"pixel-level correlation, borrowed Mog vs measured {MYELIN_LIPID}: r = {r_gl:.2f}")

# ### Cell-type territories, and which lipizones are also cell-type territories

Each matched pixel also got a cell type (majority subclass in its ball). We paint the 8 most common ones, then ask which **lipizones** co-occupy tissue with which **cell types**, using reciprocal enrichment from notebook 06.

In [ ]:
top_ct = celltype.value_counts().head(8).index.tolist()    # the 8 most common borrowed cell types get colours
pal = dict(zip(top_ct, plotting.distinct_colors(len(top_ct))))
fig, axes = plt.subplots(1, len(secs), figsize=(5.2 * len(secs), 4.4), constrained_layout=True)
for ax, s in zip(np.atleast_1d(axes), secs):
    m = (obs["SectionID"] == s).to_numpy()
    # matched pixels of less common types: a light grey backdrop
    ax.scatter(z[m & celltype.notna().to_numpy()], yv[m & celltype.notna().to_numpy()], c="0.85", s=2, rasterized=True)
    for name in top_ct:
        sel = m & (celltype == name).to_numpy()    # pixels of this section whose majority type is `name`
        ax.scatter(z[sel], yv[sel], color=pal[name], s=2.5, rasterized=True, label=name)
    cl.style.spatial_axes(ax); ax.set_title(obs.loc[m, "Condition"].iloc[0], fontsize=FS["m"])
np.atleast_1d(axes)[-1].legend(markerscale=4, loc="center left", bbox_to_anchor=(1.0, 0.5))
fig.suptitle("per-pixel cell-type territories, borrowed from MERFISH", fontsize=FS["m"])
plt.show()

In [ ]:
from scipy.cluster import hierarchy as sch
import seaborn as sns


def reciprocal_enrichment(rows, cols):
    """(col enriched within row) x (row enriched within col); > 1 means mutual over-representation."""
    # how enriched each column category is within each row category
    c1 = pd.crosstab(rows, cols); n1 = c1 / c1.sum(); n1 = (n1.T / n1.T.mean()).T
    # and the other direction
    c2 = pd.crosstab(cols, rows); n2 = c2 / c2.sum(); n2 = (n2.T / n2.T.mean()).T
    return n1 * n2.T


has_ct = celltype.notna()    # only pixels that borrowed a cell type
enr = reciprocal_enrichment(obs.loc[has_ct, "lipizone"].astype(str).rename("lipizone"),
                            celltype[has_ct].astype(str).rename("cell type"))
if enr.shape[1] > 1:
    # order cell types by similarity so related ones sit side by side
    order = sch.leaves_list(sch.linkage(sch.distance.pdist(enr.T), method="weighted", optimal_ordering=True))
    enr = enr.iloc[:, order]
# order lipizones by where their peak falls -> a near-diagonal picture
enr = enr.iloc[np.argsort(np.argmax(enr.values, axis=1)), :]

plt.figure(figsize=(10, 6))
sns.heatmap(enr, cmap="Purples", xticklabels=True, yticklabels=True, cbar_kws={"label": "reciprocal enrichment"},
            vmin=np.percentile(enr.values, 2), vmax=np.percentile(enr.values, 98))
plt.xlabel("MERFISH cell type"); plt.ylabel("lipizone"); plt.title("lipid territories vs cell-type territories")
plt.tight_layout()
plt.show()

# for each lipizone, its best-matching cell type; keep the 3 strongest pairs
best_pairs = sorted(((lz, enr.columns[enr.loc[lz].values.argmax()], float(enr.loc[lz].max())) for lz in enr.index),
                    key=lambda t: -t[2])[:3]
for lz, ct, v in best_pairs:
    print(f"lipizone {lz:>3}  <->  {ct}   (enrichment {v:.1f})")

# A blocky matrix means a lipid-defined territory is often also a cell-type territory: the cells that build a region's membranes shape its lipidome.

# Part B · Per region: do a region's genes predict its lipid change?

## Step 1: one lipid change per region

Our statistical unit changes from pixel to **Allen region**: one number per lipid per region, the pregnancy **log2 fold change** of the region's mean (pregnant / control), computed on the uMAIA layer. We keep only regions with ≥ 50 pixels in **both** conditions, so the averages mean something. By hand for the caudoputamen (`CP`) first.

In [ ]:
EPS = 1e-6    # keeps log2 finite if a mean is ~0
MIN_PX = 50    # a region needs >= 50 pixels in each condition
in_cp = (obs["acronym"].astype(str) == "CP").to_numpy()    # caudoputamen pixels
# log2(pregnant mean / control mean), one value per lipid
cp_change = np.log2((U[in_cp & ~is_ctrl].mean(0) + EPS) / (U[in_cp & is_ctrl].mean(0) + EPS))
print(f"CP: {(in_cp & is_ctrl).sum()} control vs {(in_cp & ~is_ctrl).sum()} pregnant pixels")
for jj in np.argsort(-np.abs(cp_change))[:3]:    # the 3 lipids that move most in CP
    print(f"   {lipid_names[jj]:>18s}  log2FC = {cp_change[jj]:+.2f}")

rows = {}    # region -> its row of lipid changes
for reg, ix in obs.groupby("acronym", observed=True).indices.items():    # ix = row numbers of this region's pixels
    c = is_ctrl[ix]    # which of them are control
    if c.sum() >= MIN_PX and (~c).sum() >= MIN_PX:
        rows[str(reg)] = np.log2((U[ix][~c].mean(0) + EPS) / (U[ix][c].mean(0) + EPS))
change = pd.DataFrame.from_dict(rows, orient="index", columns=lipid_names)    # regions x lipids
print("\nregion x lipid change matrix:", change.shape)
if "CP" in change.index:
    print("hand-built CP row matches the matrix:", bool(np.allclose(change.loc["CP"].to_numpy(), cp_change)))

# ## Step 2: bring in the genes, join on the region

`avemerfish_imputed_named.parquet` is the Allen MERFISH transcriptome, imputed to thousands of genes and **averaged inside each Allen region**. Its row index is the region acronym, the same key as our change matrix. The join keeps regions present in both.

In [ ]:
genes_all = pd.read_parquet(paths.tutorial_data("avemerfish_imputed_named.parquet"))
# drop a leftover pandas index column if present
genes_all = genes_all.loc[:, [c for c in genes_all.columns if c != "__index_level_0__"]]
shared = change.index.intersection(genes_all.index)    # regions present in BOTH tables
change, genes = change.loc[shared], genes_all.loc[shared]    # same regions, same order, in both
print("region x gene matrix (all):", genes_all.shape)
print(f"after the join: {len(shared)} regions | change {change.shape} | genes {genes.shape}")

# Around a hundred regions against thousands of genes is the classic "wide" problem: more features than examples, so a model can memorise the training rows and learn nothing. That's why we'll compress genes into a handful of programs first.

## Step 3: gradient-boosted trees, from the ground up

**A decision tree** is a flowchart of yes/no questions on the features ("is gene A above 0.3?"), predicting the average target at each leaf. One tree is blocky and overfits if grown deep.

**Boosting** fits a tree, measures its errors (residuals), fits a *second* tree to those residuals, adds a small fraction of it (the learning rate), and repeats. Hundreds of humble trees, each patching what came before, sum to a flexible model. Trees catch thresholds and interactions for free, which a straight line can't. Guardrails against overfitting: shallow trees (`max_depth=3`), row/column subsampling (80%), and an L2 penalty.

In [ ]:
x_toy = np.linspace(0, 1, 200)
truth = np.sin(2 * np.pi * x_toy) + 0.5 * x_toy    # a wiggly true function
y_toy = truth + rng.normal(0, 0.25, x_toy.size)    # what we observe: truth + noise
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4), sharey=True)
for ax, n_trees in zip(axes, [1, 5, 200]):
    m = xgb.XGBRegressor(n_estimators=n_trees, learning_rate=0.3, max_depth=3, random_state=SEED)    # same model, only the number of trees changes
    m.fit(x_toy.reshape(-1, 1), y_toy)
    ax.scatter(x_toy, y_toy, s=8, c="0.7", label="noisy data")
    ax.plot(x_toy, truth, "k", lw=1.2, label="truth")
    ax.plot(x_toy, m.predict(x_toy.reshape(-1, 1)), c="crimson", lw=1.6, label="XGBoost")
    ax.set_title(f"{n_trees} tree{'s' if n_trees > 1 else ''}", fontsize=FS["m"])
axes[0].legend()
plt.tight_layout()
plt.show()

# ## Step 4: genes → programs, and one regressor per lipid, with no leakage

**NMF gene programs:** MinMax-scale each gene to [0, 1] (NMF needs non-negative input), then factor regions x genes into 20 programs: `W` (regions x programs) becomes our features, `H` (programs x genes) is each program's recipe.

**Honest scoring:** split the regions once into train (75%) and held-out test (25%). The **whole feature pipeline** (MinMax, NMF, z-scoring) is fit on the training regions only, then applied to the test regions, so test regions never shape the features they're scored on. One XGBoost per lipid, scored by the **held-out Pearson r** between predicted and true change.

In [ ]:
N_PROGRAMS = int(min(20, max(2, len(shared) // 4)))     # 20 on the real data; fewer if very few regions
tr, te = train_test_split(np.arange(len(shared)), test_size=0.25, random_state=SEED)    # region indices: 75% train, 25% held out

mm = MinMaxScaler().fit(genes.values[tr])                                    # gene scaling: train regions only
Xg = np.clip(mm.transform(genes.values), 0, 1)                               # test regions clipped into the train range
nmf = NMF(n_components=N_PROGRAMS, init="nndsvda", random_state=SEED, max_iter=500).fit(Xg[tr])    # programs learned from training regions only
W = nmf.transform(Xg)                                                        # program activity for every region
H = nmf.components_    # programs x genes: each program's recipe
Wz = StandardScaler().fit(W[tr]).transform(W)                                # z-score: train regions only
prog_names = [f"program{k + 1}" for k in range(N_PROGRAMS)]    # readable names: program1, program2, ...


# one fresh, identically configured XGBoost regressor (used for every lipid)
def new_model():
    return xgb.XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=3, subsample=0.8,
                            colsample_bytree=0.8, reg_lambda=1.0, random_state=SEED, objective="reg:squarederror")


def safe_r(a, b):
    """Pearson r, or 0 if either side is constant (r is undefined there)."""
    return float(pearsonr(a, b)[0]) if np.std(a) > 0 and np.std(b) > 0 else 0.0


score_rows, shap_cols = [], {}    # held-out r per lipid, and mean |SHAP| per program per lipid
for lip in change.columns:
    yv_ = change[lip].to_numpy()    # this lipid's change in every region
    model = new_model().fit(Wz[tr], yv_[tr])    # learn: gene programs -> lipid change (training regions)
    # score on the regions the model never saw
    score_rows.append((lip, safe_r(model.predict(Wz[te]), yv_[te])))
    # SHAP (next section): XGBoost's exact TreeSHAP via pred_contribs; the last column is the bias, dropped
    contribs = model.get_booster().predict(xgb.DMatrix(Wz), pred_contribs=True)[:, :-1]
    shap_cols[lip] = np.abs(contribs).mean(0)

# best-predicted lipids first
scores = pd.DataFrame(score_rows, columns=["lipid", "test_r"]).sort_values("test_r", ascending=False)
shap_mat = pd.DataFrame(shap_cols, index=prog_names)    # programs x lipids
scores.to_csv(paths.RESULTS / "gene_model_scores.csv", index=False)
print(f"{len(tr)} train / {len(te)} test regions | {N_PROGRAMS} programs")
print(f"lipids with held-out r > 0.3: {(scores['test_r'] > 0.3).sum()} of {len(scores)} | mean r {scores['test_r'].mean():.2f}")
print(scores.head(6).to_string(index=False))

In [ ]:
best = scores.iloc[0]["lipid"]    # the best-predicted lipid, to show as a scatter
# refit it and predict the held-out regions
pred_best = new_model().fit(Wz[tr], change[best].to_numpy()[tr]).predict(Wz[te])
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
axes[0].scatter(change[best].to_numpy()[te], pred_best, s=24, c="crimson")
# a diagonal line spanning the data, for perfect prediction
lim = [min(change[best].to_numpy()[te].min(), pred_best.min()), max(change[best].to_numpy()[te].max(), pred_best.max())]
axes[0].plot(lim, lim, "--", c="0.4", lw=0.8)
axes[0].set_xlabel(f"true log2FC ({best})"); axes[0].set_ylabel("predicted")
axes[0].set_title(f"best-predicted lipid, held-out regions (r = {scores.iloc[0]['test_r']:.2f})", fontsize=FS["m"])
axes[1].hist(scores["test_r"], bins=20, color="steelblue", edgecolor="white")
axes[1].axvline(0.3, ls="--", c="crimson", lw=0.8)
axes[1].set_xlabel("held-out Pearson r"); axes[1].set_ylabel("lipids")
axes[1].set_title("how predictable is each lipid's change?", fontsize=FS["m"])
plt.tight_layout()
plt.show()

# A modest average r is the honest answer, not a failure: the gene matrix is a regional average from other mice, and a lipid is the end product of enzymes, transport and turnover, not a direct readout of one transcript. Some lipids (sometimes the headline myelin ones) aren't readable from regional gene programs at all, which is a real negative result.

## Step 5: SHAP, which program drives each prediction

**SHAP** comes from game theory: treat the 20 programs as players producing a payout (the prediction), and fairly split the payout by averaging each player's marginal contribution over every order they could join in. One signed number per program per prediction; together with a baseline they sum exactly to the prediction. For trees there's a fast exact algorithm (TreeSHAP) built into XGBoost: that's the `pred_contribs=True` above. We take mean |SHAP| per program, then average over lipids, to rank programs globally.

In [ ]:
# average each program's importance over all lipids
prog_importance = shap_mat.mean(axis=1).sort_values(ascending=False)
prog_importance.to_csv(paths.RESULTS / "gene_program_importance.csv", header=["mean_abs_shap"])
print("most predictive gene programs (mean |SHAP| across lipids):")
print(prog_importance.head(6).round(4).to_string())

fig, ax = plt.subplots(figsize=(6, 3.6))
top_p = prog_importance.head(10)[::-1]    # top 10, reversed so the biggest bar is drawn at the top
ax.barh(range(len(top_p)), top_p.values, color="mediumpurple")
ax.set_yticks(range(len(top_p))); ax.set_yticklabels(top_p.index)
ax.set_xlabel("mean |SHAP| across lipids"); ax.set_title("gene programs that predict the lipid change", fontsize=FS["m"])
plt.tight_layout()
plt.show()

# ## Step 6: the leading genes of the top program

A program is a recipe: a row of `H`, one weight per gene. The genes with the biggest weights define it.

In [ ]:
# a program's recipe is a row of H: sort its gene weights, keep the biggest
def leading_genes(prog_idx, top=30):
    return pd.Series(H[prog_idx], index=genes.columns).sort_values(ascending=False).head(top)


top_prog = prog_importance.index[0]    # the most predictive program
top_idx = prog_names.index(top_prog)    # its row number in H
lead = leading_genes(top_idx)
print(f"{top_prog} leading genes:", list(lead.head(20).index))

fig, ax = plt.subplots(figsize=(6, 4.2))
lg = lead.head(18)[::-1]    # 18 leading genes, biggest at the top
ax.barh(range(len(lg)), lg.values, color="teal")
ax.set_yticks(range(len(lg))); ax.set_yticklabels(lg.index, fontsize=FS["xs"])
ax.set_xlabel("NMF loading"); ax.set_title(f"{top_prog}: leading genes", fontsize=FS["m"])
plt.tight_layout()
plt.show()

# The most *predictive* program needn't be the tidiest biology: its top genes can be a mix of regulators, region markers and uncharacterised clones. And this is correlation across regions, not cause: to claim cause you'd need to perturb the genes.

## Step 7: gene ontology, naming the biology of a program

The **Gene Ontology** is a controlled vocabulary of biological terms (e.g. "myelination"), with every gene annotated to the terms it takes part in. **Enrichment** asks: among a program's leading genes (the *study* set), is a term over-represented compared with all genes we measured (the *background*)? A Fisher exact test per term, then Benjamini-Hochberg across the thousands of terms.

`goatools` reads the ontology graph (`go-basic.obo`) and the mouse gene → term table (`gene2go_mouse`), both in the data bundle. `mygene` converts gene symbols to the Entrez IDs those files use (one network call).

In [ ]:
from goatools.obo_parser import GODag
from goatools.anno.genetogo_reader import Gene2GoReader
from goatools.go_enrichment import GOEnrichmentStudy

with contextlib.redirect_stdout(io.StringIO()):                      # the loaders are chatty
    godag = GODag(str(paths.GO / "go-basic.obo"))
    ns2assoc = Gene2GoReader(str(paths.GO / "gene2go_mouse"), taxids=[10090]).get_ns2assc()
assoc = {}    # Entrez gene id -> set of GO terms, all namespaces merged
for d in ns2assoc.values():                                          # merge the three namespaces into one dict
    for gene_id, terms in d.items():
        assoc.setdefault(gene_id, set()).update(terms)
print(f"ontology terms: {len(godag):,} | annotated mouse genes: {len(assoc):,}")

try:
    import mygene
    mg = mygene.MyGeneInfo()    # client for the mygene.info web service

    def symbols_to_entrez(symbols):
        res = mg.querymany(list(symbols), scopes="symbol", fields="entrezgene", species="mouse", verbose=False)    # one batched web request for all symbols
        return {r["query"]: int(r["entrezgene"]) for r in res if "entrezgene" in r}    # symbol -> Entrez id, skipping symbols it couldn't map

    bg_map = symbols_to_entrez(genes.columns)                        # background = every gene we measured
    with contextlib.redirect_stdout(io.StringIO()):
        # methods=["bonferroni"] is goatools' built-in correction. We do NOT ask goatools for "fdr_bh":
        # it imports that from an old statsmodels location removed in recent releases, which would crash.
        # Instead we apply Benjamini-Hochberg ourselves (below), with the modern statsmodels function.
        goe = GOEnrichmentStudy(list(bg_map.values()), assoc, godag, propagate_counts=True,
                                alpha=0.05, methods=["bonferroni"])
    GO_OK = True    # flag for later cells: GO is available
    print(f"background genes mapped to Entrez: {len(bg_map)} of {genes.shape[1]}")
except Exception as err:                                             # offline, or mygene.info down
    GO_OK = False
    print(f"gene ontology skipped (couldn't reach mygene.info: {type(err).__name__}). Everything else still runs.")

In [ ]:
from statsmodels.stats.multitest import multipletests


def enriched_bp(prog_idx, top=100):
    """Significant (FDR < 0.05) over-represented biological-process terms for a program's leading genes."""
    study = list(symbols_to_entrez(leading_genes(prog_idx, top=top).index).values())    # the program's leading genes, as Entrez ids
    with contextlib.redirect_stdout(io.StringIO()):
        res = goe.run_study(study)    # one Fisher exact test per GO term
    bp = [r for r in res if r.NS == "BP"]                      # biological-process terms only
    if not bp:
        return pd.DataFrame(columns=["GO term (BP)", "FDR", "study_genes"])
    # Benjamini-Hochberg across every BP term tested (one Fisher p-value per term)
    fdr = multipletests([r.p_uncorrected for r in bp], method="fdr_bh")[1]
    hits = sorted(((r, q) for r, q in zip(bp, fdr) if r.enrichment == "e" and q < 0.05), key=lambda t: t[1])
    return pd.DataFrame([(r.name, q, r.ratio_in_study[0]) for r, q in hits],
                        columns=["GO term (BP)", "FDR", "study_genes"])


go_results, scan_rows = {}, []    # full GO tables per program, and one summary row per program
if GO_OK:
    for pname in prog_importance.head(8).index:                        # scan the 8 most predictive programs
        df = enriched_bp(prog_names.index(pname), top=min(100, genes.shape[1]))
        go_results[pname] = df
        scan_rows.append((pname, len(df), df["GO term (BP)"].iloc[0] if len(df) else "-",
                          float(df["FDR"].iloc[0]) if len(df) else np.nan))
# one row per scanned program
scan = pd.DataFrame(scan_rows, columns=["program", "n_sig_BP_terms", "top term", "top_FDR"])
print(scan.to_string(index=False) if len(scan) else "(no GO scan)")

# The most interpretable program: the one whose single strongest term has the lowest FDR
interp = scan.dropna(subset=["top_FDR"]).sort_values("top_FDR")
best_prog = interp.iloc[0]["program"] if len(interp) else None    # None if no program had any significant term
if best_prog:
    print(f"\nmost interpretable program: {best_prog}")
    print(go_results[best_prog].head(8).to_string(index=False))
    go_results[best_prog].to_csv(paths.RESULTS / "gene_ontology_best_program.csv", index=False)
    myelin = [g for g in ["Mobp", "Mag", "Aspa", "Plp1", "Mal", "Mog", "Mbp"] if g in genes.columns]
    ranks = leading_genes(prog_names.index(best_prog), top=genes.shape[1])    # every gene, sorted by weight
    print("classic myelin genes, rank by loading:", {g: int(ranks.index.get_loc(g)) + 1 for g in myelin})

# On real data, a clean run typically surfaces a **myelination / ensheathment** program, whose leading genes are classic oligodendrocyte transcripts. If yours surfaces a different coherent program, read its top term and its genes: that's the data-driven answer for your lipizones. The most *predictive* and the most *interpretable* program need not be the same.

## Step 8: the negative control, a permutation null

Could ~100 regions and flexible trees score this well by chance? We **break the one link we claim is real**: keep the gene programs, but **shuffle each lipid's change across regions**, refit, rescore. Repeat 20 times. If the real score sits far to the right of the shuffled cloud, the signal is real.

(Why shuffle the lipid change and not the genes? Because the claim is the region ↔ change correspondence. Shuffling the target destroys exactly that, and nothing else.)

In [ ]:
N_PERM = 20    # number of shuffles (more = finer p-value, slower)
top_lipids = scores.head(12)["lipid"].tolist()    # test the 12 best-predicted lipids


# mean held-out r over top_lipids; if perm_rng is given, shuffle each target first
def mean_test_r(perm_rng=None):
    rs = []
    for lip in top_lipids:
        yv_ = change[lip].to_numpy().copy()
        if perm_rng is not None:
            yv_ = perm_rng.permutation(yv_)                    # region A now wears region Q's change
        rs.append(safe_r(new_model().fit(Wz[tr], yv_[tr]).predict(Wz[te]), yv_[te]))
    return float(np.mean(rs))


obs_r = mean_test_r()    # the real score
prng = np.random.default_rng(0)
perm_r = np.array([mean_test_r(prng) for _ in range(N_PERM)])    # the null: N_PERM shuffled scores
p_emp = (1 + (perm_r >= obs_r).sum()) / (1 + N_PERM)    # how often chance did as well (+1s: never exactly 0)
print(f"observed mean held-out r (12 best lipids): {obs_r:.3f}")
print(f"shuffled: {perm_r.min():.3f} to {perm_r.max():.3f} (mean {perm_r.mean():.3f}) | empirical p = {p_emp:.4f}")

fig, ax = plt.subplots(figsize=(6, 3.4))
ax.hist(perm_r, bins=12, color="0.7", edgecolor="white", label="shuffled (null)")
ax.axvline(obs_r, color="crimson", lw=2, label=f"observed r = {obs_r:.2f}")
ax.set_xlabel("mean held-out Pearson r"); ax.set_ylabel("permutations"); ax.legend()
ax.set_title(f"permutation null: p = {p_emp:.3f}", fontsize=FS["m"])
plt.tight_layout()
plt.show()

# Because we picked the 12 *best* lipids before running the null, the observed number is a bit optimistic. The null still says whether region-to-region correspondence is doing real work.

## Step 9: the summary figure

All the key panels in one figure, saved as a vector PDF (text stays editable in Illustrator/Inkscape for final polishing) and a PNG.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
preg_m = (~is_ctrl) & matched    # pregnant pixels that borrowed genes (panels A and B)

ax = axes[0, 0]
ax.scatter(z[preg_m], yv[preg_m], c=lipid_vec[preg_m], cmap="plasma", s=1.5, rasterized=True,
           vmin=np.percentile(lipid_vec[matched], 2), vmax=np.percentile(lipid_vec[matched], 98))
cl.style.spatial_axes(ax); ax.set_title(f"A  {MYELIN_LIPID} (MALDI, pregnant)", loc="left")

ax = axes[0, 1]
ax.scatter(z[preg_m], yv[preg_m], c=gene_vec[preg_m], cmap="Reds", s=1.5, rasterized=True,
           vmin=np.nanpercentile(gene_vec, 2), vmax=np.nanpercentile(gene_vec, 98))
cl.style.spatial_axes(ax); ax.set_title(f"B  borrowed Mog (MERFISH), r = {r_gl:.2f}", loc="left")

ax = axes[0, 2]
ax.hist(scores["test_r"], bins=20, color="steelblue", edgecolor="white"); ax.axvline(0.3, ls="--", c="crimson", lw=0.8)
ax.set_xlabel("held-out Pearson r"); ax.set_ylabel("lipids"); ax.set_title("C  gene programs predict lipid change", loc="left")

ax = axes[1, 0]
tp = prog_importance.head(8)[::-1]
ax.barh(range(len(tp)), tp.values, color="mediumpurple"); ax.set_yticks(range(len(tp))); ax.set_yticklabels(tp.index)
ax.set_xlabel("mean |SHAP|"); ax.set_title("D  most predictive programs", loc="left")

ax = axes[1, 1]
if best_prog and len(go_results[best_prog]):
    g = go_results[best_prog].head(8)[::-1]
    ax.barh(range(len(g)), -np.log10(g["FDR"].clip(lower=1e-300)), color="darkorange")
    ax.set_yticks(range(len(g))); ax.set_yticklabels(g["GO term (BP)"], fontsize=FS["xs"])
    ax.set_xlabel("-log10 FDR"); ax.set_title(f"E  {best_prog}: enriched processes", loc="left")
else:                                                          # no GO available: show the leading genes instead
    lg = lead.head(8)[::-1]
    ax.barh(range(len(lg)), lg.values, color="teal"); ax.set_yticks(range(len(lg))); ax.set_yticklabels(lg.index)
    ax.set_xlabel("NMF loading"); ax.set_title(f"E  {top_prog}: leading genes", loc="left")

ax = axes[1, 2]
ax.hist(perm_r, bins=12, color="0.7", edgecolor="white"); ax.axvline(obs_r, color="crimson", lw=2)
ax.set_xlabel("mean held-out r"); ax.set_title(f"F  permutation null (p = {p_emp:.3f})", loc="left")

fig.suptitle("Which genes explain the pregnancy lipid changes?", fontsize=FS["l"])
plt.tight_layout()
saved = paths.savefig(fig, "fig_genes_explain_lipid_changes")    # -> figures/...pdf (editable) and .png
plt.show()
print("saved:", *[p.name for p in saved])

# ## Recap

- Two experiments that never touched the same tissue, joined through **shared Allen coordinates**, at two zoom levels.
- **Per pixel:** a KD-tree ball query borrows MERFISH genes and cell types; a borrowed myelin gene tracks a measured myelin lipid, and lipizones line up with cell-type territories.
- **Per region:** lipid change ~ 20 NMF gene programs via **XGBoost**, scored on held-out regions with **no leakage**, explained by **SHAP**, named by **gene ontology**, checked with a **permutation null**.
- Not proof of cause: a strong, testable hypothesis about which genes to perturb next.
- Tables in `results/`, the summary figure in `figures/`.

**Next (09):** your own question.